# SwinV2 HPO Shard Runner — 9 Machines × 3 Cases = 27 Total

Notebook này chạy **một shard** trong tổng số 9 shard HPO cho SwinV2-Base-384.
Mỗi Kaggle session chọn một `SHARD` khác nhau (0–8) để **không trùng** config.

## Phân công máy
| Máy | SHARD | Nhóm config |
|-----|-------|-------------|
| 1 | 0 | LR=5e-5, T_0 ngắn-trung, wd=1e-4 |
| 2 | 1 | LR=3-8e-5, biến thể LR trung bình |
| 3 | 2 | Warmup dài wu=3, LR=3-5e-5 |
| 4 | 3 | LR nhỏ (1e-5/5e-6), backbone_lr thấp |
| 5 | 4 | LR=2e-5/5e-6, backbone_lr thấp-cao |
| 6 | 5 | LR cao 1e-4, wd/blr biến thể |
| 7 | 6 | LR cao 1e-4, wd cao/blr thấp |
| 8 | 7 | AdamW beta variations |
| 9 | 8 | BACKBONE_LR_SCALE biên (0.02–0.20) |

## Config ép cứng (không thể override từ preset)
```
MODEL_TYPE      = swinv2_base_384
IMAGE_SIZE      = 384
BATCH_SIZE      = 8
FREEZE_STRATEGY = none   ← toàn bộ backbone được train
HEAD_TYPE       = ordinal
AUGMENT_VERSION = v1     ← chỉ basic spatial augmentation
```

## Scheduler
**LinearWarmup(`WARMUP_EPOCHS`) → CosineAnnealingWarmRestarts(`T_0`)** — xử lý tự động trong `train.py` khi `is_transformer=True`.

## W&B
- Mỗi case = 1 W&B run riêng, log **train/loss, train/qwk, val/loss, val/qwk, lr/head, lr/backbone** theo từng epoch.
- `group = swin_shardN` → gom tất cả run của shard vào 1 group trên W&B dashboard.
- Sau khi shard xong: summary table + bar chart QWK được log vào run `shard{N}_summary`.

**Thứ tự chạy**: Cell 1 (setup) → Cell 2 (verify config) → Cell 3 (list case) → Cell 4 (chạy shard) → Cell 5 (xem kết quả)

In [ ]:
%%time
# ── Cell 1: Repo + dependencies + W&B secret ─────────────────────────────────
import os
import subprocess
import sys

ON_KAGGLE = os.path.isdir("/kaggle/working")

if ON_KAGGLE:
    os.chdir("/kaggle/working")
    REPO_NAME = "fundus-disease-detection"
    REPO_URL  = "https://github.com/tuan8p/fundus-disease-detection.git"
    BRANCH    = "triet-optuna"
    if not os.path.isdir(REPO_NAME):
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_NAME], check=True)
    else:
        subprocess.run(["git", "-C", REPO_NAME, "pull", "origin", BRANCH], check=False)
    os.chdir(REPO_NAME)
else:
    cwd = os.getcwd()
    if os.path.basename(cwd) == "notebooks":
        cwd = os.path.dirname(cwd)
    os.chdir(cwd)
    if not os.path.isfile(os.path.join(os.getcwd(), "scripts", "run_optuna_shard_swin.py")):
        raise RuntimeError(
            "Không thấy scripts/run_optuna_shard_swin.py — cwd phải là thư mục repo."
        )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)

REPO_ROOT = os.path.abspath(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

WANDB_API_KEY = "" #Lấy key từ Kaggle Secrets
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY
        print("✓ W&B: API key loaded từ Kaggle Secrets.")
    except Exception as _e:
        print(f"⚠ W&B: secret không có hoặc lỗi ({_e}) → W&B sẽ bị disabled.")
else:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "")
    if WANDB_API_KEY:
        print("✓ W&B: API key loaded từ env WANDB_API_KEY.")
    else:
        print("⚠ W&B: WANDB_API_KEY chưa set → W&B disabled. Set env var nếu muốn log.")

print(f"REPO_ROOT = {REPO_ROOT}")
assert os.path.isfile(
    os.path.join(REPO_ROOT, "scripts", "run_optuna_shard_swin.py")
), "Thiếu scripts/run_optuna_shard_swin.py"

In [ ]:
# ── Cell 2: Kiểm tra toàn bộ config TRƯỚC KHI CHẠY ──────────────────────────
import sys
sys.path.insert(0, REPO_ROOT)

from scripts.optuna_shard_presets_swin import SHARD_PRESETS, NUM_SHARDS
from scripts.run_optuna_shard_swin import build_base_cfg, merge_case

print(f"{'='*70}")
print(f"CONFIG VERIFICATION — {NUM_SHARDS} shards × 3 cases = {sum(len(s) for s in SHARD_PRESETS)} total")
print(f"{'='*70}")

FORCED_CONFIGS = {
    "MODEL_TYPE":       "swinv2_base_384",
    "IMAGE_SIZE":       384,
    "FREEZE_STRATEGY":  "none",
    "HEAD_TYPE":        "ordinal",
    "AUGMENT_VERSION":  "v1",
}

all_ok = True
base = build_base_cfg(data_dir="/tmp", repo_root=REPO_ROOT, seed=42, shard_idx=0)

for shard_idx, shard in enumerate(SHARD_PRESETS):
    print(f"\nShard {shard_idx} ({len(shard)} cases):")
    for c in shard:
        cfg, label = merge_case(base, c)
        errors = []
        for key, expected in FORCED_CONFIGS.items():
            actual = cfg.get(key)
            if actual != expected:
                errors.append(f"{key}={actual!r} (expected {expected!r})")
        status = "✓" if not errors else "✗ ERRORS: " + "; ".join(errors)
        if errors:
            all_ok = False
        print(
            f"  {label:<38}"
            f" LR={cfg['LR']:.1e}"
            f" blr={cfg['BACKBONE_LR_SCALE']}"
            f" wd={cfg['WEIGHT_DECAY']:.0e}"
            f" wu={cfg['WARMUP_EPOCHS']}"
            f" T0={cfg['T_0']}"
            f"  {status}"
        )

print(f"\n{'='*70}")
if all_ok:
    print("✓ TẤT CẢ CONFIG HỢP LỆ — FREEZE_STRATEGY=none, AUGMENT_VERSION=v1")
else:
    print("✗ CÓ LỖI CONFIG — kiểm tra lại trước khi chạy!")
print(f"{'='*70}")

In [ ]:
%%time
# ── Cell 3 (tuỳ chọn): In danh sách tất cả shard + case ─────────────────────
import subprocess, sys, os
script = os.path.join(REPO_ROOT, "scripts", "run_optuna_shard_swin.py")
subprocess.run([sys.executable, script, "--list"], cwd=REPO_ROOT, check=True)

In [ ]:
%%time
# ── Cell 4: Chạy một shard ───────────────────────────────────────────────────
import os, subprocess, sys

SHARD            = 0   
DATA_DIR         = "/kaggle/input/datasets/trietdeptrai/aptos-preprocessed/aptos_clahe/aptos_clahe"
OUTPUT_BASE      = "/kaggle/working/optuna_shard_swin_out"
SEED             = 42
BATCH_SIZE       = 4   
GRAD_ACCUM_STEPS = 2   

os.makedirs(OUTPUT_BASE, exist_ok=True)

script = os.path.join(REPO_ROOT, "scripts", "run_optuna_shard_swin.py")
cmd = [
    sys.executable, script,
    "--shard",       str(SHARD),
    "--data-dir",    DATA_DIR,
    "--output-base", OUTPUT_BASE,
    "--seed",        str(SEED),
    "--repo-root",   REPO_ROOT,
    "--wandb-key",   WANDB_API_KEY,
    "--batch-size",  str(BATCH_SIZE),
    "--grad-accum",  str(GRAD_ACCUM_STEPS),
]


display_cmd = [c if c != WANDB_API_KEY or not WANDB_API_KEY else "<WANDB_KEY>" for c in cmd]
print("Chạy:", " ".join(display_cmd))
print(f"W&B project: optuna-fundus-shard-swin | group: swin_shard{SHARD}")
print(f"Output dir : {OUTPUT_BASE}/optuna_shards_swin/shard_{SHARD}/")
print("─" * 60)

proc = subprocess.run(cmd, cwd=REPO_ROOT)
if proc.returncode != 0:
    raise RuntimeError(f"run_optuna_shard_swin.py thoát với mã {proc.returncode}")

summary = os.path.join(
    OUTPUT_BASE, "optuna_shards_swin", f"shard_{SHARD}", "shard_summary.json"
)
print("\nShard hoàn tất!")
print(f"Summary JSON: {summary}")
print(f"Tồn tại     : {os.path.isfile(summary)}")

In [ ]:
# ── Cell 5: Xem kết quả shard ─────────────────────────────────────────────────
import json, os
summary_path = os.path.join(
    OUTPUT_BASE, "optuna_shards_swin", f"shard_{SHARD}", "shard_summary.json"
)
with open(summary_path) as f:
    s = json.load(f)

print(f"Model  : {s['model']}")
print(f"Shard  : {s['shard']}")
print(f"Best   : {s['best_label']}  QWK={s['best_max_val_qwk']:.4f}")
print()
print(f"{'Label':<42} {'QWK':>7}  Config")
print("-" * 80)
for c in sorted(s["cases"], key=lambda x: x["max_val_qwk"], reverse=True):
    snap = c.get("cfg_snapshot", {})
    err  = "  ← ERROR" if c["error"] else ""
    detail = (
        f"  lr={snap.get('LR', '-'):.1e}"
        f"  blr={snap.get('BACKBONE_LR_SCALE', '-')}"
        f"  wd={snap.get('WEIGHT_DECAY', '-'):.0e}"
        f"  wu={snap.get('WARMUP_EPOCHS', '-')}"
        f"  T0={snap.get('T_0', '-')}"
    )
    print(f"{c['label']:<42} {c['max_val_qwk']:>7.4f}{detail}{err}")

In [ ]:
# ── Cell 6 (tuỳ chọn): Verify W&B runs đã được log ──────────────────────────
import os, json
shard_dir   = os.path.join(OUTPUT_BASE, "optuna_shards_swin", f"shard_{SHARD}")

print(f"Checking W&B meta files in shard_{SHARD}...\n")
found = 0
for case_dir in sorted(os.listdir(shard_dir)):
    full = os.path.join(shard_dir, case_dir)
    if not os.path.isdir(full):
        continue
    meta_path    = os.path.join(full, "wandb_run_meta.json")
    history_path = os.path.join(full, "history.json")
    has_meta    = os.path.isfile(meta_path)
    has_history = os.path.isfile(history_path)

    wandb_url = ""
    if has_meta:
        with open(meta_path) as f:
            meta = json.load(f)
        wandb_url = meta.get("url", "")
        found += 1

    best_qwk = ""
    if has_history:
        with open(history_path) as f:
            hist = json.load(f)
        vq = hist.get("val_qwk", [])
        best_qwk = f"best_val_qwk={max(vq):.4f}" if vq else "no qwk data"

    wb_status = f"W&B ✓  {wandb_url}" if has_meta else "W&B ✗ (disabled or failed)"
    print(f"  {case_dir}")
    print(f"    history: {'✓' if has_history else '✗'}  {best_qwk}")
    print(f"    {wb_status}")

print(f"\n{found}/{len([d for d in os.listdir(shard_dir) if os.path.isdir(os.path.join(shard_dir, d))])} cases có W&B run meta.")